# Wavelength - Model Development

Exploratory notebook for the Wavelength mood classifier.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load processed data

In [ ]:
df = pd.read_parquet('../backend/app/data/processed_tracks.parquet')
print(f'Shape: {df.shape}')
print(df.dtypes)

## Class distribution

In [ ]:
dist = df['mood'].value_counts()
print(dist)

fig, ax = plt.subplots(figsize=(8, 4))
dist.plot(kind='bar', ax=ax, color='#d97706', edgecolor='none')
ax.set_title('Mood label distribution')
ax.set_xlabel('')
ax.set_ylabel('Track count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Feature correlations

In [ ]:
feature_cols = [
    'acousticness', 'danceability', 'energy', 'instrumentalness',
    'liveness', 'loudness_norm', 'speechiness', 'tempo_norm', 'valence',
    'energy_valence_ratio', 'acoustic_energy_contrast',
    'danceability_tempo_score', 'mood_index'
]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[feature_cols].corr()
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

## Energy vs Valence by mood

In [ ]:
MOOD_COLORS = {
    'Happy': '#92400e',
    'Energetic': '#991b1b',
    'Melancholic': '#4c1d95',
    'Focused': '#064e3b',
    'Calm': '#1e3a8a',
    'Intense': '#374151',
    'Neutral': '#9ca3af',
}

fig, ax = plt.subplots(figsize=(9, 6))
sample = df.sample(min(5000, len(df)), random_state=42)

for mood, color in MOOD_COLORS.items():
    mask = sample['mood'] == mood
    ax.scatter(sample.loc[mask, 'energy'], sample.loc[mask, 'valence'],
               c=color, label=mood, alpha=0.5, s=8, linewidths=0)

ax.set_xlabel('Energy')
ax.set_ylabel('Valence')
ax.set_title('Energy vs Valence by mood')
ax.legend(markerscale=2, fontsize=9)
plt.tight_layout()
plt.show()

## Model metrics

In [ ]:
import json

with open('../backend/app/model/metrics.json') as f:
    metrics = json.load(f)

print(f"Best model: {metrics['best_model']}")
for name, m in metrics['models'].items():
    print(f"{name}: F1={m['f1_macro']:.4f}  acc={m['accuracy']:.4f}")